In [ ]:
import sys
import os
sys.path.append(os.path.abspath("..")) 

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import Column, Integer, String, DateTime, func
from sqlalchemy.orm import declarative_base
from sqlalchemy import inspect
from sqlalchemy import text
from database import engine

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from database import Base, SessionLocal, engine, ensure_views_from_files, init_db
from main.sql import load_dict, count_overlap_word
from main.utils import get_completion

In [ ]:
init_db()
ensure_views_from_files()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS TranslationLog"))

## Update Word Rarity Project

In [ ]:

# ── Step 1: Find cutoff ─────────────────────────────────────────────────────
# Words added before the first Uncommon entry were only rated under a 2-category
# system (Common / Rare) and need their rarity re-evaluated.

full_df = load_dict()  # columns use display names, e.g. 'Word Rarity', 'Added Date'

first_uncommon_date = (
    full_df.loc[full_df['Word Rarity'] == 'Uncommon', 'Added Date']
    .min()
)
print(f"First 'Uncommon' word added on: {first_uncommon_date}")

pre_uncommon_df = full_df.loc[full_df['Added Date'] < first_uncommon_date].copy()
print(f"Words requiring rarity review: {len(pre_uncommon_df)}")
pre_uncommon_df[['Word Id', 'Word', 'Word Rarity', 'Added Date']].head(10)


### Review Log
Create (or reload) a CSV that tracks whether each pre-Uncommon word has already been processed through the 3-category rarity update.  Re-running this cell is safe — it only appends missing rows, never overwrites `reviewed` flags already set.

In [ ]:

import os

REVIEW_LOG_PATH = os.path.join(os.path.dirname(os.getcwd()), "Repo", "GPT4LanguageLearning", "rarity_review_log.csv")
# Fall back: same directory as the notebook
REVIEW_LOG_PATH = os.path.join(os.path.abspath(".."), "rarity_review_log.csv")

def build_or_reload_review_log(pre_uncommon_df: pd.DataFrame, log_path: str) -> pd.DataFrame:
    """
    Build a review log for words that need 3-category rarity re-evaluation.

    If the log CSV already exists:
      - Keep all existing rows (preserving the `reviewed` flag).
      - Append any words present in `pre_uncommon_df` that are NOT yet in the log.
    If the log CSV does not exist:
      - Create a fresh log from `pre_uncommon_df` with `reviewed = False`.

    Columns: word_id | word | old_rarity | new_rarity | reviewed
    """
    new_rows = pre_uncommon_df[['Word Id', 'Word', 'Word Rarity']].copy()
    new_rows.columns = ['word_id', 'word', 'old_rarity']
    new_rows['new_rarity'] = pd.NA
    new_rows['reviewed'] = False

    if os.path.exists(log_path):
        existing_log = pd.read_csv(log_path)
        already_tracked = set(existing_log['word_id'])
        to_append = new_rows.loc[~new_rows['word_id'].isin(already_tracked)]
        if len(to_append):
            print(f"Appending {len(to_append)} new words to existing review log.")
            review_log = pd.concat([existing_log, to_append], ignore_index=True)
        else:
            print("Review log is up-to-date; no new rows to append.")
            review_log = existing_log
    else:
        print(f"Creating new review log with {len(new_rows)} words.")
        review_log = new_rows

    review_log.to_csv(log_path, index=False)
    return review_log


review_log = build_or_reload_review_log(pre_uncommon_df, REVIEW_LOG_PATH)
print(f"\nReview log shape: {review_log.shape}")
print(f"Pending review : {(~review_log['reviewed']).sum()}")
print(f"Already reviewed: {review_log['reviewed'].sum()}")
review_log.head(10)


### Rarity-only SQL update helper
A targeted `UPDATE` that only touches the `word_rarity` column — quiz history and all other fields are left untouched.

In [ ]:

from main.translation import get_prompt_for_multiclass_rarity_classification
from main.utils import parse_response_table
import time

def sql_update_word_rarity(word_rarity_map: dict) -> int:
    """
    Update `word_rarity` in WordDict for the given words without touching any
    other column (quiz stats remain intact).

    Parameters
    ----------
    word_rarity_map : dict
        {word: new_rarity_string}  e.g. {'你好': 'Common', '菊花': 'Uncommon'}

    Returns
    -------
    int  Number of rows actually updated.
    """
    if not word_rarity_map:
        return 0

    updated = 0
    with engine.begin() as conn:
        for word, new_rarity in word_rarity_map.items():
            result = conn.execute(
                text("UPDATE WordDict SET word_rarity = :rarity WHERE word = :word"),
                {"rarity": new_rarity, "word": word}
            )
            updated += result.rowcount

    print(f"Updated word_rarity for {updated} row(s).")
    return updated


### Batch rarity update runner
Processes unreviewed words in configurable batch sizes, calls the GPT 3-category classifier, applies targeted DB updates, and persists progress to the review log after every batch.

In [ ]:

def run_batch_rarity_update(
    log_path: str,
    batch_size: int = 30,
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    dry_run: bool = False,
) -> pd.DataFrame:
    """
    Process unreviewed words through the 3-category rarity classifier and update
    the database, one batch at a time.

    Parameters
    ----------
    log_path    : str   Path to the review log CSV.
    batch_size  : int   Number of words per GPT call (default 30).
    model       : str   OpenAI model to use.
    temperature : float Sampling temperature.
    dry_run     : bool  If True, print results without writing to the DB or log.

    Returns
    -------
    pd.DataFrame  The updated review log.
    """
    review_log = pd.read_csv(log_path)
    pending = review_log.loc[~review_log['reviewed']].copy()

    if pending.empty:
        print("Nothing to process — all words already reviewed.")
        return review_log

    print(f"Total pending: {len(pending)} words across "
          f"{-(-len(pending) // batch_size)} batch(es) of {batch_size}.")

    for batch_start in range(0, len(pending), batch_size):
        batch = pending.iloc[batch_start: batch_start + batch_size]
        word_list = batch['word'].tolist()
        batch_num = batch_start // batch_size + 1
        print(f"\n── Batch {batch_num} ({len(word_list)} words) ──────────────────")

        # ── Call GPT ──────────────────────────────────────────────────────────
        try:
            prompt = get_prompt_for_multiclass_rarity_classification(word_list)
            response = get_completion(prompt, model=model, temperature=temperature)
            rarity_df = parse_response_table(response.choices[0].message.content)
        except Exception as e:
            print(f"  !! GPT call failed: {e}  — skipping batch.")
            continue

        if 'Word Rarity' not in rarity_df.columns or 'Word' not in rarity_df.columns:
            print("  !! Unexpected response format — skipping batch.")
            print(rarity_df)
            continue

        rarity_df = rarity_df[['Word', 'Word Rarity']].dropna()
        rarity_df.columns = ['word', 'new_rarity']

        word_rarity_map = dict(zip(rarity_df['word'], rarity_df['new_rarity']))
        print(f"  Classified {len(word_rarity_map)} word(s):")
        for w, r in word_rarity_map.items():
            print(f"    {w:10s}  →  {r}")

        # ── Apply DB update ───────────────────────────────────────────────────
        if not dry_run:
            sql_update_word_rarity(word_rarity_map)

            # Persist progress to the review log
            for word, new_rarity in word_rarity_map.items():
                mask = review_log['word'] == word
                review_log.loc[mask, 'new_rarity'] = new_rarity
                review_log.loc[mask, 'reviewed'] = True

            review_log.to_csv(log_path, index=False)
            reviewed_so_far = review_log['reviewed'].sum()
            print(f"  Progress saved. Reviewed so far: {reviewed_so_far}/{len(review_log)}")
        else:
            print("  [dry_run=True] — DB and log NOT updated.")

        time.sleep(0.5)   # gentle pacing between batches

    print("\n── Done ─────────────────────────────────────────────────────────")
    remaining = (~review_log['reviewed']).sum()
    print(f"Remaining unreviewed: {remaining}")
    return review_log


In [ ]:

# ── Run the batch update ─────────────────────────────────────────────────────
# Set dry_run=True to preview GPT output without touching the DB or log.
# Set dry_run=False when ready to commit changes.

review_log = run_batch_rarity_update(
    log_path=REVIEW_LOG_PATH,
    batch_size=30,
    model="gpt-4o-mini",
    temperature=0.0,
    dry_run=True,      # ← flip to False to apply changes
)
review_log
